# Derivation of Hamiltonian and Necessary Conditions

In this notebook, we consider the following periodic optimal control problem:

$$
    \begin{align*}
        \max_{u} \quad & R = \frac{1}{T_{\text{lap}}(u)} \int_{0}^{T_{\text{lap}}(u)} \sum_{j=1}^N \gamma_j q_j(t) \, dt           \\
        \text{s.t.} \quad & \dot{s} = u(t)\\
        & s(0)=0,\; s(T_{\text{lap}}(u)) = L\\
        & \dot{q}_j(t) = S_j(s)(1 - q_j(t))^2 - \alpha q_j(t)^2, \nonumber\\ &\quad \forall j \in \{1, \dots, N\}\\
        & q_j(0) = q_j(T_{\text{lap}}(u)), \quad \forall j \in \{1, \dots, N\} \\
        & \frac{1}{T_{\text{lap}}(u)} \int_{0}^{T_{\text{lap}}(u)} P_{\mathrm{out}}(u(t)) \, dt = P_{\mathrm{in,avg}}.
    \end{align*}
$$

We begin by transcribing the objective and dynamics into Julia. We will be using the `Symbolics.jl` package for this analysis.

In [1]:
using Symbolics, Plots, LaTeXStrings, Latexify

We define all the variables for the problem:

In [2]:
# Define dimension N (keep it small for ease of derivation)
N_nodes = 2
@variables t

# Define states, costates, and control
@variables s(t) b(t) u(t)
@variables λ_s(t) λ_b(t)

# Define array states (q) and costates (lambda_q)
@variables q[1:N_nodes] λ_q[1:N_nodes]

# Define parameters
@variables P_in_avg α t_f
@variables γ[1:N_nodes]

# Define functions for P_out and S
@variables P_out(..) S(..)[1:N_nodes]


2-element Vector{Symbolics.CallAndWrap}:
 P_out⋆
 S⋆

We define the objective:

In [8]:
# Running objective
println("Objective function:")
g_x = (1 / t_f) * sum(γ[i] * q[i] for i in 1:N_nodes)

Objective function:


(q[1]*γ[1] + q[2]*γ[2]) / t_f

In [13]:
println(latexify(g_x))

\begin{equation}
\frac{q_{1} ~ \gamma_{1} + q_{2} ~ \gamma_{2}}{\mathtt{t\_f}}
\end{equation}



We define all the dynamics:

In [27]:
ṡ = u
ḃ = P_in_avg - P_out(u)

q̇ = [S(s)[i] * (1 - q[i])^2 - α * q[i]^2 for i in 1:N_nodes]
println(latexify(q̇))
display(latexify(q̇))

\begin{equation}
\left[
\begin{array}{c}
\left( 1 - q_{1} \right)^{2} ~ S\_{1}\left( s\left( t \right) \right) - \left( q_{1} \right)^{2} ~ \alpha \\
\left( 1 - q_{2} \right)^{2} ~ S\_{2}\left( s\left( t \right) \right) - \left( q_{2} \right)^{2} ~ \alpha \\
\end{array}
\right]
\end{equation}



L"\begin{equation}
\left[
\begin{array}{c}
\left( 1 - q_{1} \right)^{2} ~ S\_{1}\left( s\left( t \right) \right) - \left( q_{1} \right)^{2} ~ \alpha \\
\left( 1 - q_{2} \right)^{2} ~ S\_{2}\left( s\left( t \right) \right) - \left( q_{2} \right)^{2} ~ \alpha \\
\end{array}
\right]
\end{equation}
"

## Constructing the Hamiltonian

In [28]:
# Inner product of costates and dynamics 
inner_product = λ_s * ṡ + λ_b * ḃ + sum(λ_q[i] * q̇[i] for i in 1:N_nodes)

H = g_x + inner_product

println("=== Hamiltonian ===")
println("H = ", H)
println()
println("H = ", latexify(H))
display(latexify(H))

=== Hamiltonian ===
H = (q[1]*γ[1] + q[2]*γ[2]) / t_f + (P_in_avg - P_out(u(t)))*λ_b(t) + u(t)*λ_s(t) + ((S(s(t)))[1]*((1 - q[1])^2) - (q[1]^2)*α)*λ_q[1] + ((S(s(t)))[2]*((1 - q[2])^2) - (q[2]^2)*α)*λ_q[2]

H = \begin{equation}
\frac{q_{1} ~ \gamma_{1} + q_{2} ~ \gamma_{2}}{\mathtt{t\_f}} + \left( \mathtt{P\_in\_avg} - \mathtt{P\_out}\left( u\left( t \right) \right) \right) ~ \mathtt{\lambda\_b}\left( t \right) + u\left( t \right) ~ \mathtt{\lambda\_s}\left( t \right) + \left( \left( 1 - q_{1} \right)^{2} ~ S\_{1}\left( s\left( t \right) \right) - \left( q_{1} \right)^{2} ~ \alpha \right) ~ \mathtt{\lambda\_q}_{1} + \left( \left( 1 - q_{2} \right)^{2} ~ S\_{2}\left( s\left( t \right) \right) - \left( q_{2} \right)^{2} ~ \alpha \right) ~ \mathtt{\lambda\_q}_{2}
\end{equation}



"\\begin{equation}\n\\frac{q_{1} ~ \\gamma_{1} + q_{2} ~ \\gamma_{2}}{\\mathtt{t\\_f}} + \\left( \\mathtt{P\\_in\\_avg} - \\mathtt{P\\_out}\\left( u\\left( t \\right) \\right) \\right) ~ \\mathtt{\\lambda\\_b}\\left( t \\right) + u\\left( t \\right) ~ \\mathtt{\\lambda\\_s}\\left( t \\right) + \\left( " ⋯ 31 bytes ⋯ "S\\_{1}\\left( s\\left( t \\right) \\right) - \\left( q_{1} \\right)^{2} ~ \\alpha \\right) ~ \\mathtt{\\lambda\\_q}_{1} + \\left( \\left( 1 - q_{2} \\right)^{2} ~ S\\_{2}\\left( s\\left( t \\right) \\right) - \\left( q_{2} \\right)^{2} ~ \\alpha \\right) ~ \\mathtt{\\lambda\\_q}_{2}\n\\end{equation}\n"

## Deriving First-Order Necessary Conditions

In [21]:
λ_ṡ = -Symbolics.derivative(H, s)
λ_ḃ = -Symbolics.derivative(H, b)
λ_q̇ = [-Symbolics.derivative(H, q[i]) for i in 1:N_nodes]

# Control Optimality: dH/du = 0
dH_du = Symbolics.derivative(H, u)

λ_s(t) - Differential(u(t), 1)(P_out(u(t)))*λ_b(t)

In [26]:
println("=== Costate Dynamics ===")
println("d(λ_s)/dt = ", λ_ṡ)
display(latexify(λ_ṡ))
println("d(λ_s)/dt = ", latexify(λ_ṡ))

println("d(λ_b)/dt = ", λ_ḃ)   # Outputs 0 -> lambda_b is constant
display(latexify(λ_ḃ))
println("d(λ_b)/dt = ", latexify(λ_ḃ))


for i in 1:N_nodes
    println("d(λ_q_$i)/dt = ", λ_q̇[i])
    display(latexify(λ_q̇[i]))
    println("d(λ_q_$i)/dt = ", latexify(λ_q̇[i]))
end
println()

L"\begin{equation}
 - \left( 1 - q_{1} \right)^{2} ~ \frac{\mathrm{d} ~ S\_{1}\left( s\left( t \right) \right)}{\mathrm{d}s(t)} ~ \mathtt{\lambda\_q}_{1} - \left( 1 - q_{2} \right)^{2} ~ \frac{\mathrm{d} ~ S\_{2}\left( s\left( t \right) \right)}{\mathrm{d}s(t)} ~ \mathtt{\lambda\_q}_{2}
\end{equation}
"

L"\begin{equation}
0
\end{equation}
"

=== Costate Dynamics ===
d(λ_s)/dt = -Differential(s(t), 1)((S(s(t)))[1])*((1 - q[1])^2)*λ_q[1] - Differential(s(t), 1)((S(s(t)))[2])*((1 - q[2])^2)*λ_q[2]
d(λ_s)/dt = \begin{equation}
 - \left( 1 - q_{1} \right)^{2} ~ \frac{\mathrm{d} ~ S\_{1}\left( s\left( t \right) \right)}{\mathrm{d}s(t)} ~ \mathtt{\lambda\_q}_{1} - \left( 1 - q_{2} \right)^{2} ~ \frac{\mathrm{d} ~ S\_{2}\left( s\left( t \right) \right)}{\mathrm{d}s(t)} ~ \mathtt{\lambda\_q}_{2}
\end{equation}

d(λ_b)/dt = 0
d(λ_b)/dt = \begin{equation}
0
\end{equation}

d(λ_q_1)/dt = -(γ[1] / t_f) - (-2(S(s(t)))[1]*(1 - q[1]) - 2q[1]*α)*λ_q[1]
d(λ_q_1)/dt = \begin{equation}
 - \frac{\gamma_{1}}{\mathtt{t\_f}} - \left(  - 2 ~ S\_{1}\left( s\left( t \right) \right) ~ \left( 1 - q_{1} \right) - 2 ~ q_{1} ~ \alpha \right) ~ \mathtt{\lambda\_q}_{1}
\end{equation}

d(λ_q_2)/dt = -(γ[2] / t_f) - (-2(S(s(t)))[2]*(1 - q[2]) - 2q[2]*α)*λ_q[2]
d(λ_q_2)/dt = \begin{equation}
 - \frac{\gamma_{2}}{\mathtt{t\_f}} - \left(  - 2 ~ S\_{2}\left( s\l

L"\begin{equation}
 - \frac{\gamma_{1}}{\mathtt{t\_f}} - \left(  - 2 ~ S\_{1}\left( s\left( t \right) \right) ~ \left( 1 - q_{1} \right) - 2 ~ q_{1} ~ \alpha \right) ~ \mathtt{\lambda\_q}_{1}
\end{equation}
"

L"\begin{equation}
 - \frac{\gamma_{2}}{\mathtt{t\_f}} - \left(  - 2 ~ S\_{2}\left( s\left( t \right) \right) ~ \left( 1 - q_{2} \right) - 2 ~ q_{2} ~ \alpha \right) ~ \mathtt{\lambda\_q}_{2}
\end{equation}
"

In [25]:
println("=== Control Optimality Condition (dH/du = 0) ===")
println("dH/du = ", dH_du)
display(latexify(dH_du))
println("dH/du = ", latexify(dH_du))
println()

L"\begin{equation}
\mathtt{\lambda\_s}\left( t \right) - \frac{\mathrm{d} ~ \mathtt{P\_out}\left( u\left( t \right) \right)}{\mathrm{d}u(t)} ~ \mathtt{\lambda\_b}\left( t \right)
\end{equation}
"

=== Control Optimality Condition (dH/du = 0) ===
dH/du = λ_s(t) - Differential(u(t), 1)(P_out(u(t)))*λ_b(t)
dH/du = \begin{equation}
\mathtt{\lambda\_s}\left( t \right) - \frac{\mathrm{d} ~ \mathtt{P\_out}\left( u\left( t \right) \right)}{\mathrm{d}u(t)} ~ \mathtt{\lambda\_b}\left( t \right)
\end{equation}


